# BIST AI Radar V8.2 Colab
V5 dönem kilitleme + V8 skor motoru birleşik sürüm.

In [ ]:
%cd /content
!rm -rf bist-ai-radar
!git clone https://github.com/ErgunUral/bist-ai-radar.git
%cd /content/bist-ai-radar
!pip install -r requirements.txt -q


In [ ]:
import pandas as pd, time
from datetime import datetime
from src.bist_provider import BistProvider
from src.analyzer import analyze_statements
from src.sheets_dashboard import prepare_main_dashboard, top_n, risky_stocks, safe_update_worksheet
from src.period_manager import period_keys_from_tables, period_label_from_key, find_column_by_period


In [ ]:
SPREADSHEET_ID = "1nZJmVeFVwNo5NLSIQ3eFu5A_-r0bto4qFsek68hYSqE"
PORTFOY_TXT = "ALFAS,ANELE,ASELS,BALSU,BIENY,BINHO,BNTAS,BYDNR,DGGYO,DOAS,ECILC,EKOS,ERBOS,ERCB,ESEN,INGRM,ISCTR,IZINV,KARTN,KBORU,KONKA,KTSKR,KZGYO,MAVI,MEGAP,MERIT,NIBAS,OZGYO,PGSUS,PKENT,PRKAB,RAYSG,RGYAS,SISE,SMRTG,TARKM,ULKER,VERTU,VRGYO"
PORTFOY = [x.strip().upper() for x in PORTFOY_TXT.split(",") if x.strip()]
print(len(PORTFOY), "hisse")


In [ ]:
provider = BistProvider()
rows, period_rows, failed = [], [], []
for i,ticker in enumerate(PORTFOY,1):
    print(f"[{i}/{len(PORTFOY)}] {ticker}", end=" ")
    try:
        st = provider.fetch(ticker)
        keys = period_keys_from_tables(st.balance_sheet, st.income_statement, st.cash_flow, limit=12)
        if not keys:
            print("veri yok"); failed.append(ticker); continue
        latest = keys[0]
        res = analyze_statements(ticker, st)
        d = res.to_dict()
        rows.append({
            "Hisse": res.ticker, "Dönem": res.period, "Sektör Tipi": res.sector_type,
            "Genel Skor": res.total_score, "AI Karar": res.decision, "Risk": res.risk,
            "AI Uyarı": res.warnings, "ROE %": d.get("roe"), "ROA %": d.get("roa"),
            "Net Kar Marjı %": d.get("net_margin"), "FAVÖK Marjı %": d.get("ebitda_margin"),
            "Net Borç": d.get("net_debt"), "Cari Oran": d.get("current_ratio"),
            "Borç/Özkaynak": d.get("debt_to_equity"), "F/K": d.get("price_to_earnings"),
            "PD/DD": d.get("price_to_book"), "FD/FAVÖK": d.get("ev_to_ebitda"),
            "Son Güncelleme": datetime.now().strftime("%d.%m.%Y %H:%M")
        })
        period_rows.append({
            "Hisse": ticker, "En Güncel Dönem": period_label_from_key(latest),
            "Mevcut Dönemler": ", ".join(period_label_from_key(k) for k in keys),
            "Bilanço En Güncel Kolon": str(find_column_by_period(st.balance_sheet, latest) or ""),
            "Gelir En Güncel Kolon": str(find_column_by_period(st.income_statement, latest) or ""),
            "Nakit Akış En Güncel Kolon": str(find_column_by_period(st.cash_flow, latest) or "")
        })
        print("OK", res.period, res.decision)
    except Exception as e:
        print("HATA", str(e)[:80]); failed.append(ticker)
    time.sleep(.8)
dashboard = prepare_main_dashboard(pd.DataFrame(rows))
period_log = pd.DataFrame(period_rows)
print("Başarılı:", len(dashboard), "Hatalı:", failed)
dashboard


In [ ]:
display(top_n(dashboard,20))
display(risky_stocks(dashboard,40))
display(period_log)


In [ ]:
from google.colab import auth
import gspread
from google.auth import default
auth.authenticate_user()
creds,_ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_key(SPREADSHEET_ID)
if dashboard is None or dashboard.empty:
    raise ValueError("Dashboard boş geldi. Sheets temizlenmedi.")
print(safe_update_worksheet(sh, "Göstergeler", dashboard, backup=True))
print(safe_update_worksheet(sh, "Dönem Kontrol", period_log, backup=True))
print(sh.url)
